In [2]:
import os, json, math 

HERE = os.path.abspath('') 

HEX_DIR = os.path.join(HERE, 'golden_out') 

In [3]:
with open(os.path.join(HEX_DIR, 'params.json')) as f:
    params = json.load(f)


In [9]:
IN_BITS = params['in_bits'] 
ABITS = params['abits'] 
W_FRAC = params['w_frac'] 
M_BITS = params['m_bits'] 
BQ_BITS = params['bq_bits'] 
AMAX = (1 << ABITS) -1 
INMAX = (1 << IN_BITS) -1 
SH = params['SH'] 
pools = params['pools'] 
kept_idx = params['kept_idx'] 
T = params['T'] 
nL = len(SH) 

print('Params loaded from params.json:')
print(f'  Layers : {nL}')
print(f'  Pools  : {pools}')
print(f'  IN_BITS={IN_BITS}, ABITS={ABITS}, W_FRAC={W_FRAC}')
print(f'  M_BITS={M_BITS}, BQ_BITS={BQ_BITS}, AMAX={AMAX}')
print(f'  Decision threshold T={T}')
print(f'  Kept channels per layer: {[len(k) for k in kept_idx]}')
print(f'  SH values: {SH}')

Params loaded from params.json:
  Layers : 5
  Pools  : [True, True, False, True, False]
  IN_BITS=4, ABITS=6, W_FRAC=7
  M_BITS=16, BQ_BITS=30, AMAX=63
  Decision threshold T=2688
  Kept channels per layer: [8, 16, 23, 31, 29]
  SH values: [19, 21, 22, 22, 23]


In [7]:
def read_hex_unsigned(path): 
    with open(path) as f: 
        lines = [l.strip() for l in f if l.strip()] 
    return [int(x,16) for x in lines] 

In [29]:
def sign_extend(value, bits): 
    if value >= (1 << (bits-1)): 
        value -= (1 << bits) 
    return value 

In [11]:
def read_hex_signed(path, bits): 
    values = read_hex_unsigned(path) 
    return [sign_extend(v,bits) for v in values] 

In [12]:
sample = read_hex_unsigned(os.path.join(HEX_DIR, 'img0_in.hex'))
print(f'img0_in.hex: first 8 values = {sample[:8]}')
print(f'Total pixels: {len(sample)} (should be 64x64 = {64*64})')
print(f'Pixel value range: [{min(sample)}, {max(sample)}]  (expect [0, {INMAX}])')

img0_in.hex: first 8 values = [9, 9, 9, 9, 9, 9, 9, 9]
Total pixels: 4096 (should be 64x64 = 4096)
Pixel value range: [0, 14]  (expect [0, 15])


In [13]:
def make_grid(flat, H, W): 
    assert len(flat) == H * W, f'Expected {H*W} values, got {len(flat)}' 
    return [[flat[r*W+c] for c in range(W)] for r in range(H)] 

In [14]:
def make_tensor3d(flat, C, H, W): 
    assert len(flat) == C*H*W, f"Expected {C*H*W} values, got {len(flat)}" 
    return [[[

        flat[c*H*W + r * W + col] for col in range(W)
    ] for r in range(H)] for c in range (C)]

In [21]:
def make_tensor4d(flat, C_out, C_in, KH, KW): 
    assert len(flat) == C_out * C_in * KH * KW, f"Expected {C_out * C_in *KH * KW}, got {len(flat)}" 
    stride_co = C_in * KH * KW
    stride_ci = KH * KW 
    stride_kh = KW 

    return [[[[ flat[co * stride_co + ci *stride_ci + kh * stride_kh + kw] 
              for kw in range(KW)]
            for kh in range(KH)]
            for ci in range(C_in)] 
           for co in range(C_out)]

In [22]:
#weight encoding style 
#4 bit -> [s e2 e1 e0] 
#with s the sign and power of two encodig
#For example 1000 -> 0 
#Next example 1100 -> 64  
#Another example 0100 -> 16
#Another example 

In [33]:
def decode_weight_code(code):
    """
    Decode a 4-bit pow2 weight code [s e2 e1 e0] to an integer weight.

    Bit layout:
      bit 3  = s (sign: 0=positive, 1=negative)
      bits 2:0 = e (exponent, 3 bits)

    Result:
      e == 0  ->  0
      e != 0  ->  (-1)^s * 2^(W_FRAC - e)   where W_FRAC=7
    """
    s = (code >> 3) & 1   # sign bit
    e = code & 7           # 3-bit exponent
    if e == 0:
        return 0
    magnitude = 1 << (W_FRAC - e)   # 2^(7-e)
    return -magnitude if s else magnitude

print('Weight decode table (all 16 codes):') 

print('Weight decode table (all 16 codes):')
print(f'  {"code":>4}  {"binary":>6}  {"s":>2}  {"e":>2}  {"w_int":>6}')
for code in range(16):
    s = (code >> 3) & 1
    e = code & 7
    w = decode_weight_code(code)
    print(f'  {code:>4}  {bin(code)[2:].zfill(4):>6}  {s:>2}  {e:>2}  {w:>6}')

    

Weight decode table (all 16 codes):
Weight decode table (all 16 codes):
  code  binary   s   e   w_int
     0    0000   0   0       0
     1    0001   0   1      64
     2    0010   0   2      32
     3    0011   0   3      16
     4    0100   0   4       8
     5    0101   0   5       4
     6    0110   0   6       2
     7    0111   0   7       1
     8    1000   1   0       0
     9    1001   1   1     -64
    10    1010   1   2     -32
    11    1011   1   3     -16
    12    1100   1   4      -8
    13    1101   1   5      -4
    14    1110   1   6      -2
    15    1111   1   7      -1


In [26]:
W_int  = []   # decoded integer weights: list of 4D arrays [cout][cin][3][3]
W_code = []   # raw 4-bit codes:         list of 4D arrays [cout][cin][3][3]

print()
for i in range(nL):
    cout = len(kept_idx[i])
    cin  = 1 if i == 0 else len(kept_idx[i - 1])

    codes_flat = read_hex_unsigned(os.path.join(HEX_DIR, f'w_L{i}.hex'))
    codes_4d   = make_tensor4d(codes_flat, cout, cin, 3, 3)
    W_code.append(codes_4d)

    # Decode every code to its integer weight
    wint_4d = [[[[decode_weight_code(codes_4d[co][ci][ky][kx])
                  for kx in range(3)]
                 for ky in range(3)]
                for ci in range(cin)]
               for co in range(cout)]
    W_int.append(wint_4d)

    # Sample: show first kernel of first output channel
    sample_kernel = codes_4d[0][0]  # shape [3][3] of codes
    sample_wint   = wint_4d[0][0]   # shape [3][3] of integers
    print(f'Layer {i}: weight shape [{cout}][{cin}][3][3]')
    print(f'  code[0][0] kernel : {sample_kernel}')
    print(f'  wint[0][0] kernel : {sample_wint}')



Layer 0: weight shape [8][1][3][3]
  code[0][0] kernel : [[13, 1, 11], [9, 2, 10], [11, 1, 10]]
  wint[0][0] kernel : [[-4, 64, -16], [-64, 32, -32], [-16, 64, -32]]
Layer 1: weight shape [16][8][3][3]
  code[0][0] kernel : [[15, 9, 10], [10, 9, 9], [13, 9, 9]]
  wint[0][0] kernel : [[-1, -64, -32], [-32, -64, -64], [-4, -64, -64]]
Layer 2: weight shape [23][16][3][3]
  code[0][0] kernel : [[10, 10, 10], [10, 10, 4], [9, 11, 10]]
  wint[0][0] kernel : [[-32, -32, -32], [-32, -32, -8], [-64, -16, -32]]
Layer 3: weight shape [31][23][3][3]
  code[0][0] kernel : [[2, 6, 4], [10, 10, 3], [6, 10, 14]]
  wint[0][0] kernel : [[32, -2, -8], [-32, -32, 16], [-2, -32, -2]]
Layer 4: weight shape [29][31][3][3]
  code[0][0] kernel : [[10, 11, 10], [6, 11, 9], [4, 10, 9]]
  wint[0][0] kernel : [[-32, -16, -32], [-2, -16, -64], [-8, -32, -64]]


In [27]:
n_classes = 2
n_fc_in   = len(kept_idx[-1])

fc_codes_flat = read_hex_unsigned(os.path.join(HEX_DIR, 'fc_w.hex'))
# FC layout: [class][cin], stored row-major
FC_code = [[fc_codes_flat[cl * n_fc_in + ci] for ci in range(n_fc_in)]
            for cl in range(n_classes)]
FC_int  = [[decode_weight_code(FC_code[cl][ci]) for ci in range(n_fc_in)]
            for cl in range(n_classes)]
print(f'\nFC: weight shape [{n_classes}][{n_fc_in}]')
print(f'  FC_int[0][:5] = {FC_int[0][:5]}')
print(f'  FC_int[1][:5] = {FC_int[1][:5]}')


FC: weight shape [2][29]
  FC_int[0][:5] = [-64, -64, 64, -64, 64]
  FC_int[1][:5] = [64, 64, -32, 64, -32]


In [30]:
BN_M  = []   # per-layer list[cout], signed 16-bit
BN_Bq = []   # per-layer list[cout], signed BQ_BITS-bit

print()
for i in range(nL):
    cout = len(kept_idx[i])
    m_vals  = read_hex_signed(os.path.join(HEX_DIR, f'bn_M_L{i}.hex'),  M_BITS)
    bq_vals = read_hex_signed(os.path.join(HEX_DIR, f'bn_B_L{i}.hex'),  BQ_BITS)
    assert len(m_vals)  == cout, f'Layer {i}: expected {cout} M values, got {len(m_vals)}'
    assert len(bq_vals) == cout, f'Layer {i}: expected {cout} Bq values, got {len(bq_vals)}'
    BN_M.append(m_vals)
    BN_Bq.append(bq_vals)
    print(f'Layer {i}: BN M[0]={m_vals[0]:8d}, Bq[0]={bq_vals[0]:12d}  (cout={cout})')



Layer 0: BN M[0]=   17814, Bq[0]=      -24928  (cout=8)
Layer 1: BN M[0]=   11915, Bq[0]=    39232058  (cout=16)
Layer 2: BN M[0]=   11527, Bq[0]=   112546174  (cout=23)
Layer 3: BN M[0]=   22877, Bq[0]=   269206370  (cout=31)
Layer 4: BN M[0]=   16528, Bq[0]=    38533624  (cout=29)


In [51]:
def conv3x3_code(inp, w_code, cin, cout, H, W):
    H_pad = H + 2
    W_pad = W + 2
    inp_pad = [[[0] * W_pad for _ in range(H_pad)] for _ in range(cin)]

    for c in range(cin):
        for r in range(H):
            for col in range(W):
                inp_pad[c][r + 1][col + 1] = inp[c][r][col]

    out = [[[0] * W for _ in range(H)] for _ in range(cout)]
    for co in range(cout):
        for r in range(H):
            for col in range(W):
                acc = 0
                for ci in range(cin):
                    for ky in range(3):
                        for kx in range(3):
                            w = w_code[co][ci][ky][kx]
                            e = w & 7
                            if e == 0:
                                continue  # Trọng số bằng 0 -> bỏ qua MAC
                            pixel = inp_pad[ci][r + ky][col + kx]
                            val = pixel << (W_FRAC - e)
                            if (w >> 3) & 1:
                                acc -= val
                            else:
                                acc += val
                out[co][r][col] = acc
    return out



In [45]:
def requant(acc_map, M_list, Bq_list, SH_VAL, cout, H,W): 
    out = [[[0] * W for _ in range(H)] for _ in range(cout)]
    for co in range(cout):
        m  = M_list[co]
        bq = Bq_list[co]
        for r in range(H):
            for col in range(W):
                # 1. BN: scale accumulator and add bias
                val = acc_map[co][r][col] * m + bq #Must be a multiplier/DSP
                # 2. Arithmetic right-shift to bring into 6-bit range
                val = val >> SH_VAL
                # 3. ReLU + 6-bit saturation: clamp to [0, 63]
                val = max(0, min(AMAX, val))
                out[co][r][col] = val
    return out
    

In [46]:
def maxpool2(inp, cout, H, W): 
    H_out = H//2 
    W_out = W//2 

    out   = [[[0] * W_out for _ in range(H_out)] for _ in range(cout)] 
    for c in range(cout): 
        for r in range(H_out):
            for col in range(W_out): 
                r0 = r * 2
                c0 = col * 2
                val = max(
                    inp[c][r0    ][c0    ],
                    inp[c][r0    ][c0 + 1],
                    inp[c][r0 + 1][c0    ],
                    inp[c][r0 + 1][c0 + 1]
                )
                out[c][r][col] = val
    return out 

In [47]:
def global_avg_pool_sum(inp, cout, H, W):
    
    S = [0] * cout
    for c in range(cout):
        total = 0
        for r in range(H):
            for col in range(W):
                total += inp[c][r][col]
        S[c] = total
    return S

In [52]:

def fc_layer(S, fc_code, n_classes, n_in):
    acc_fc = [0] * n_classes
    for cl in range(n_classes):
        acc = 0
        for c in range(n_in):
            fc = fc_code[cl][c]
            e = fc & 7
            if e == 0:
                continue  # Trọng số bằng 0 -> bỏ qua
            val = S[c] << (W_FRAC - e)
            if (fc >> 3) & 1:
                acc -= val
            else:
                acc += val
        acc_fc[cl] = acc
    return acc_fc

In [53]:
def golden_forward(img_in, H=64, W=64):
    """
    Full CNN forward pass on a single image using pure Python integers.

    Args:
        img_in : list[H][W]  — 4-bit input pixels, values 0..15
        H, W   : int         — spatial dimensions (default 64x64)

    Returns:
        dict with keys:
          'layer_accs'    : list[nL] of list[cout][H][W]  — raw conv accumulators
          'layer_outputs' : list[nL] of list[cout][H][W]  — 6-bit activations (after requant & pool)
          'S'             : list[cout_last]               — GAP spatial sums
          'n_pixels'      : int                           — H*W at last layer
          'acc_fc'        : list[2]                       — FC class scores
          'D'             : int                           — decision value
          'pred'          : int                           — 1=person, 0=no_person
    """
    # Layer 0 input: wrap single-channel image as [1][H][W]
    act = [[img_in[r][:] for r in range(H)]]   # list[1][H][W]
    H_cur, W_cur = H, W

    layer_outputs = []
    layer_accs    = []

    for i in range(nL):
        cout = len(kept_idx[i])
        cin  = len(act)

        print(f'  Layer {i}: [{cin}x{H_cur}x{W_cur}]', end='')

        # ---- 1. 3x3 Convolution ----
        acc = conv3x3_code(act, W_code[i], cin, cout, H_cur, W_cur)
        layer_accs.append(acc)

        # ---- 2. Requantise: BN + ReLU + 6-bit clamp ----
        act = requant(acc, BN_M[i], BN_Bq[i], SH[i], cout, H_cur, W_cur)

        # ---- 3. Optional 2x2 MaxPool ----
        if pools[i]:
            act = maxpool2(act, cout, H_cur, W_cur)
            H_cur //= 2
            W_cur //= 2

        layer_outputs.append(act)
        pool_str = 'pool' if pools[i] else 'no pool'
        print(f' -> [{cout}x{H_cur}x{W_cur}]  ({pool_str})')

    # ---- 4. Global Average Pool (sum) ----
    cout_last = len(kept_idx[-1])
    S         = global_avg_pool_sum(act, cout_last, H_cur, W_cur)
    n_pixels  = H_cur * W_cur
    print(f'  GAP: [{cout_last}x{H_cur}x{W_cur}] -> S[{cout_last}]  (n_pixels={n_pixels})')

    # ---- 5. Fully Connected ----
    acc_fc = fc_layer(S, FC_code, n_classes=2, n_in=cout_last)
    print(f'  FC : acc_fc = {acc_fc}')

    # ---- 6. Decision ----
    D    = acc_fc[1] - acc_fc[0]
    pred = 1 if D > T else 0
    print(f'  Decision: D={D}, T={T}, pred={pred}')

    return {
        'layer_accs'    : layer_accs,
        'layer_outputs' : layer_outputs,
        'S'             : S,
        'n_pixels'      : n_pixels,
        'acc_fc'        : acc_fc,
        'D'             : D,
        'pred'          : pred,
    }

print('golden_forward defined')

golden_forward defined


In [54]:
# Select the first test image described in params.json
IMG_IDX = params['images'][0]['idx']    # e.g. 0
LABEL   = params['images'][0]['label']  # 0 = no_person, 1 = person

in_flat = read_hex_unsigned(os.path.join(HEX_DIR, f'img{IMG_IDX}_in.hex'))
H_in = W_in = 64
img_in = make_grid(in_flat, H_in, W_in)   # nested list [H][W], values 0..15

label_str = 'person' if LABEL else 'no_person'
print(f'Image index : {IMG_IDX}')
print(f'Label       : {LABEL} ({label_str})')
print(f'Shape       : {H_in} x {W_in}')
print(f'Value range : [{min(in_flat)}, {max(in_flat)}]  (expect [0, {INMAX}])')
print()
print('Top-left 8x8 corner of the image:')
for row in img_in[:8]:
    print(' ', ' '.join(f'{v:2d}' for v in row[:8]))
print(f'Running golden model on image {IMG_IDX}  (label={LABEL})...')
print()
result = golden_forward(img_in)
print()

label_str = 'person (pred=1)' if result['pred'] == 1 else 'no_person (pred=0)'
ref_D     = params['images'][0].get('D', 'N/A')

print('=' * 50)
print(f"Result for image {IMG_IDX} (true label={LABEL}):")
print(f"  acc_fc          = {result['acc_fc']}")
print(f"  D = acc_fc[1] - acc_fc[0] = {result['D']}")
print(f"  T (threshold)   = {T}")
print(f"  Prediction      : {label_str}")
if ref_D != 'N/A':
    match = result['D'] == ref_D
    print(f"  Expected D (params.json): {ref_D}")
    print(f"  D match                 : {match}")
print('=' * 50)

Image index : 0
Label       : 0 (no_person)
Shape       : 64 x 64
Value range : [0, 14]  (expect [0, 15])

Top-left 8x8 corner of the image:
   9  9  9  9  9  9  9  9
   9  9  9  9  9  9  9  9
   9  9  9  9  9  9  9  9
   9  9  9  9  9  9  9  9
   9  9  9  9  9  9  9  9
   9  9  9  9  9  9  9  9
   9  9  9  9  9  9  9  9
   9  9  9  9  9  9  9  7
Running golden model on image 0  (label=0)...

  Layer 0: [1x64x64] -> [8x32x32]  (pool)
  Layer 1: [8x32x32] -> [16x16x16]  (pool)
  Layer 2: [16x16x16] -> [23x16x16]  (no pool)
  Layer 3: [23x16x16] -> [31x8x8]  (pool)
  Layer 4: [31x8x8] -> [29x8x8]  (no pool)
  GAP: [29x8x8] -> S[29]  (n_pixels=64)
  FC : acc_fc = [-14976, -11776]
  Decision: D=3200, T=2688, pred=1

Result for image 0 (true label=0):
  acc_fc          = [-14976, -11776]
  D = acc_fc[1] - acc_fc[0] = 3200
  T (threshold)   = 2688
  Prediction      : person (pred=1)
  Expected D (params.json): 3200
  D match                 : True


In [56]:
#!/usr/bin/env python3
"""Sinh cnn_params.h va test_vectors.h tu thu muc golden_out/ cho cnn.c (Bambu HLS).

Dung:  python3 gen_params.py [golden_out] [out_dir]
"""
import json
import os
import sys


def rd(path):
    with open(path, "r", encoding="utf-8") as f:
        return [int(l.strip(), 16) for l in f if l.strip()]


def sx(v, bits):
    return v - (1 << bits) if v >= (1 << (bits - 1)) else v


def emit(lines, ctype, name, vals):
    lines.append(f"static const {ctype} {name}[{len(vals)}] = {{")
    for i in range(0, len(vals), 16):
        lines.append("  " + ", ".join(str(v) for v in vals[i:i + 16]) + ",")
    lines.append("};")
    lines.append("")


def main():
    hex_dir = "golden_out"
    out_dir = "."
    with open(os.path.join(hex_dir, "params.json"), "r", encoding="utf-8") as f:
        p = json.load(f)

    kept_idx = p["kept_idx"]
    n = len(kept_idx)
    assert p["m_bits"] <= 16 and p["bq_bits"] <= 32
    H0 = 64

    head = [
        "/* auto-generated by gen_params.py - do not edit */",
        "#ifndef CNN_PARAMS_H",
        "#define CNN_PARAMS_H",
        "#include <stdint.h>",
        "",
        f"#define IMG_H {H0}",
        f"#define IMG_W {H0}",
        f"#define N_LAYERS {n}",
        f"#define W_FRAC {p['w_frac']}",
        f"#define AMAX {(1 << p['abits']) - 1}",
        f"#define THRESH {p['T']}",
    ]
    body = []
    buf = H0 * H0
    cur = H0
    cout = 1
    for i in range(n):
        cout = len(kept_idx[i])
        cin = 1 if i == 0 else len(kept_idx[i - 1])
        pool = 1 if p["pools"][i] else 0
        nxt = cur // 2 if pool else cur
        buf = max(buf, cin * cur * cur, cout * nxt * nxt)

        head += [
            f"#define L{i}_CIN {cin}",
            f"#define L{i}_COUT {cout}",
            f"#define L{i}_H {cur}",
            f"#define L{i}_SH {p['SH'][i]}",
            f"#define L{i}_POOL {pool}",
        ]
        w = rd(os.path.join(hex_dir, f"w_L{i}.hex"))
        assert len(w) == cout * cin * 9, f"w_L{i}: {len(w)} != {cout * cin * 9}"
        m = [sx(v, p["m_bits"]) for v in rd(os.path.join(hex_dir, f"bn_M_L{i}.hex"))]
        b = [sx(v, p["bq_bits"]) for v in rd(os.path.join(hex_dir, f"bn_B_L{i}.hex"))]
        assert len(m) == cout and len(b) == cout
        emit(body, "uint8_t", f"L{i}_W", w)      # flat [co][ci][ky][kx]
        emit(body, "int16_t", f"L{i}_M", m)
        emit(body, "int32_t", f"L{i}_B", b)
        cur = nxt

    head += [f"#define GAP_C {cout}", f"#define GAP_H {cur}", f"#define BUF_SIZE {buf}", ""]
    fc = rd(os.path.join(hex_dir, "fc_w.hex"))
    assert len(fc) == 2 * cout
    emit(body, "uint8_t", "FC_W", fc)           # flat [class][c]

    with open(os.path.join(out_dir, "cnn_params.h"), "w") as f:
        f.write("\n".join(head + [""] + body + ["#endif", ""]))

    # test vectors
    tv = ["/* auto-generated by gen_params.py */", "#include <stdint.h>", ""]
    imgs = p["images"]
    for k, im in enumerate(imgs):
        px = rd(os.path.join(hex_dir, f"img{im['idx']}_in.hex"))
        assert len(px) == H0 * H0
        emit(tv, "uint8_t", f"IMG{k}", px)
    tv.append(f"#define TV_N {len(imgs)}")
    tv.append("static const uint8_t *const TV_IMG[TV_N] = { "
              + ", ".join(f"IMG{k}" for k in range(len(imgs))) + " };")
    tv.append("static const int32_t TV_D[TV_N] = { "
              + ", ".join(str(im["D"]) for im in imgs) + " };")
    tv.append("static const int TV_PRED[TV_N] = { "
              + ", ".join(str(im["pred"]) for im in imgs) + " };")
    tv.append("")
    with open(os.path.join(out_dir, "test_vectors.h"), "w") as f:
        f.write("\n".join(tv))
    print(f"OK: BUF_SIZE={buf}, GAP={cout}x{cur}x{cur}, {len(imgs)} test images")


if __name__ == "__main__":
    main()

OK: BUF_SIZE=8192, GAP=29x8x8, 2 test images


In [57]:
#!/usr/bin/env python3
"""Sinh cnn_params.h va test_vectors.h tu thu muc golden_out/ cho cnn.c (Bambu HLS).

Dung:  python3 gen_params.py [golden_out] [out_dir]
"""
import json
import os
import sys


def rd(path):
    with open(path, "r", encoding="utf-8") as f:
        return [int(l.strip(), 16) for l in f if l.strip()]


def sx(v, bits):
    return v - (1 << bits) if v >= (1 << (bits - 1)) else v


def pack(codes):
    """Pack 4-bit codes, 8 per uint32_t (code k at bits [4*(k%8) +: 4] of word k//8)."""
    assert all(0 <= c < 16 for c in codes)
    out = []
    for i in range(0, len(codes), 8):
        w = 0
        for j, c in enumerate(codes[i:i + 8]):
            w |= c << (4 * j)
        out.append(w)
    return out


def emit(lines, ctype, name, vals):
    lines.append(f"static const {ctype} {name}[{len(vals)}] = {{")
    for i in range(0, len(vals), 16):
        lines.append("  " + ", ".join(str(v) for v in vals[i:i + 16]) + ",")
    lines.append("};")
    lines.append("")


def main():
    hex_dir = "golden_out"
    out_dir = "."
    with open(os.path.join(hex_dir, "params.json"), "r", encoding="utf-8") as f:
        p = json.load(f)

    kept_idx = p["kept_idx"]
    n = len(kept_idx)
    assert p["m_bits"] <= 16 and p["bq_bits"] <= 32
    H0 = 64

    head = [
        "/* auto-generated by gen_params.py - do not edit */",
        "#ifndef CNN_PARAMS_H",
        "#define CNN_PARAMS_H",
        "#include <stdint.h>",
        "",
        f"#define IMG_H {H0}",
        f"#define IMG_W {H0}",
        f"#define N_LAYERS {n}",
        f"#define W_FRAC {p['w_frac']}",
        f"#define AMAX {(1 << p['abits']) - 1}",
        f"#define THRESH {p['T']}",
    ]
    body = []
    buf_a = H0 * H0   # layer-0 input (image) lives in A
    buf_b = 0
    cur = H0
    cout = 1
    for i in range(n):
        cout = len(kept_idx[i])
        cin = 1 if i == 0 else len(kept_idx[i - 1])
        pool = 1 if p["pools"][i] else 0
        nxt = cur // 2 if pool else cur
        in_sz, out_sz = cin * cur * cur, cout * nxt * nxt
        if i % 2 == 0:   # even layer: A -> B
            buf_a, buf_b = max(buf_a, in_sz), max(buf_b, out_sz)
        else:            # odd layer: B -> A
            buf_b, buf_a = max(buf_b, in_sz), max(buf_a, out_sz)

        head += [
            f"#define L{i}_CIN {cin}",
            f"#define L{i}_COUT {cout}",
            f"#define L{i}_H {cur}",
            f"#define L{i}_HW {cur * cur}",
            f"#define L{i}_SH {p['SH'][i]}",
            f"#define L{i}_POOL {pool}",
        ]
        w = rd(os.path.join(hex_dir, f"w_L{i}.hex"))
        assert len(w) == cout * cin * 9, f"w_L{i}: {len(w)} != {cout * cin * 9}"
        m = [sx(v, p["m_bits"]) for v in rd(os.path.join(hex_dir, f"bn_M_L{i}.hex"))]
        b = [sx(v, p["bq_bits"]) for v in rd(os.path.join(hex_dir, f"bn_B_L{i}.hex"))]
        assert len(m) == cout and len(b) == cout
        emit(body, "uint32_t", f"L{i}_W", pack(w))   # packed codes, flat [co][ci][ky][kx]
        emit(body, "int16_t", f"L{i}_M", m)
        emit(body, "int32_t", f"L{i}_B", b)
        cur = nxt

    head += [f"#define GAP_C {cout}", f"#define GAP_H {cur}", f"#define BUF_A_SIZE {buf_a}", f"#define BUF_B_SIZE {buf_b}", ""]
    fc = rd(os.path.join(hex_dir, "fc_w.hex"))
    assert len(fc) == 2 * cout
    emit(body, "uint32_t", "FC_W", pack(fc))    # packed codes, flat [class][c]

    with open(os.path.join(out_dir, "cnn_params.h"), "w") as f:
        f.write("\n".join(head + [""] + body + ["#endif", ""]))

    # test vectors
    tv = ["/* auto-generated by gen_params.py */", "#include <stdint.h>", ""]
    imgs = p["images"]
    for k, im in enumerate(imgs):
        px = rd(os.path.join(hex_dir, f"img{im['idx']}_in.hex"))
        assert len(px) == H0 * H0
        emit(tv, "uint8_t", f"IMG{k}", px)
    tv.append(f"#define TV_N {len(imgs)}")
    tv.append("static const uint8_t *const TV_IMG[TV_N] = { "
              + ", ".join(f"IMG{k}" for k in range(len(imgs))) + " };")
    tv.append("static const int32_t TV_D[TV_N] = { "
              + ", ".join(str(im["D"]) for im in imgs) + " };")
    tv.append("static const int TV_PRED[TV_N] = { "
              + ", ".join(str(im["pred"]) for im in imgs) + " };")
    tv.append("")
    with open(os.path.join(out_dir, "test_vectors.h"), "w") as f:
        f.write("\n".join(tv))
    print(f"OK: BUF_A={buf_a} BUF_B={buf_b}, GAP={cout}x{cur}x{cur}, {len(imgs)} test images")


if __name__ == "__main__":
    main()

OK: BUF_A=4096 BUF_B=8192, GAP=29x8x8, 2 test images


In [58]:
#!/usr/bin/env python3
"""Sinh cnn_params.h va test_vectors.h tu thu muc golden_out/ cho cnn.c (Bambu HLS).

Dung:  python3 gen_params.py [golden_out] [out_dir]
"""
import json
import os
import sys


def rd(path):
    with open(path, "r", encoding="utf-8") as f:
        return [int(l.strip(), 16) for l in f if l.strip()]


def sx(v, bits):
    return v - (1 << bits) if v >= (1 << (bits - 1)) else v


def pack(codes):
    """Pack 4-bit codes, 8 per uint32_t (code k at bits [4*(k%8) +: 4] of word k//8)."""
    assert all(0 <= c < 16 for c in codes)
    out = []
    for i in range(0, len(codes), 8):
        w = 0
        for j, c in enumerate(codes[i:i + 8]):
            w |= c << (4 * j)
        out.append(w)
    return out


def emit(lines, ctype, name, vals):
    lines.append(f"static const {ctype} {name}[{len(vals)}] = {{")
    for i in range(0, len(vals), 16):
        lines.append("  " + ", ".join(str(v) for v in vals[i:i + 16]) + ",")
    lines.append("};")
    lines.append("")


def main():
    hex_dir = "golden_out"
    out_dir = "."
    with open(os.path.join(hex_dir, "params.json"), "r", encoding="utf-8") as f:
        p = json.load(f)

    kept_idx = p["kept_idx"]
    n = len(kept_idx)
    assert p["m_bits"] <= 16 and p["bq_bits"] <= 32
    H0 = 64

    head = [
        "/* auto-generated by gen_params.py - do not edit */",
        "#ifndef CNN_PARAMS_H",
        "#define CNN_PARAMS_H",
        "#include <stdint.h>",
        "",
        f"#define IMG_H {H0}",
        f"#define IMG_W {H0}",
        f"#define N_LAYERS {n}",
        f"#define W_FRAC {p['w_frac']}",
        f"#define AMAX {(1 << p['abits']) - 1}",
        f"#define THRESH {p['T']}",
    ]
    body = []
    w_all, m_all, b_all = [], [], []   # one ROM each (fewer separate memories for Bambu)
    buf_a = H0 * H0   # layer-0 input (image) lives in A
    buf_b = 0
    cur = H0
    cout = 1
    for i in range(n):
        cout = len(kept_idx[i])
        cin = 1 if i == 0 else len(kept_idx[i - 1])
        pool = 1 if p["pools"][i] else 0
        nxt = cur // 2 if pool else cur
        in_sz, out_sz = cin * cur * cur, cout * nxt * nxt
        if i % 2 == 0:   # even layer: A -> B
            buf_a, buf_b = max(buf_a, in_sz), max(buf_b, out_sz)
        else:            # odd layer: B -> A
            buf_b, buf_a = max(buf_b, in_sz), max(buf_a, out_sz)

        head += [
            f"#define L{i}_CIN {cin}",
            f"#define L{i}_COUT {cout}",
            f"#define L{i}_H {cur}",
            f"#define L{i}_HW {cur * cur}",
            f"#define L{i}_SH {p['SH'][i]}",
            f"#define L{i}_POOL {pool}",
        ]
        w = rd(os.path.join(hex_dir, f"w_L{i}.hex"))
        assert len(w) == cout * cin * 9, f"w_L{i}: {len(w)} != {cout * cin * 9}"
        m = [sx(v, p["m_bits"]) for v in rd(os.path.join(hex_dir, f"bn_M_L{i}.hex"))]
        b = [sx(v, p["bq_bits"]) for v in rd(os.path.join(hex_dir, f"bn_B_L{i}.hex"))]
        assert len(m) == cout and len(b) == cout
        head += [f"#define L{i}_WOFF {len(w_all)}", f"#define L{i}_MOFF {len(m_all)}"]
        w_all += pack(w)   # packed codes, flat [co][ci][ky][kx], padded to a word per layer
        m_all += m
        b_all += b
        cur = nxt

    head += [f"#define GAP_C {cout}", f"#define GAP_H {cur}", f"#define BUF_A_SIZE {buf_a}", f"#define BUF_B_SIZE {buf_b}", ""]
    fc = rd(os.path.join(hex_dir, "fc_w.hex"))
    assert len(fc) == 2 * cout
    head.append(f"#define FC_WOFF {len(w_all)}")
    w_all += pack(fc)                           # packed codes, flat [class][c]
    emit(body, "uint32_t", "W_ALL", w_all)
    emit(body, "int16_t", "M_ALL", m_all)
    emit(body, "int32_t", "B_ALL", b_all)

    with open(os.path.join(out_dir, "cnn_params.h"), "w") as f:
        f.write("\n".join(head + [""] + body + ["#endif", ""]))

    # test vectors
    tv = ["/* auto-generated by gen_params.py */", "#include <stdint.h>", ""]
    imgs = p["images"]
    for k, im in enumerate(imgs):
        px = rd(os.path.join(hex_dir, f"img{im['idx']}_in.hex"))
        assert len(px) == H0 * H0
        emit(tv, "uint8_t", f"IMG{k}", px)
    tv.append(f"#define TV_N {len(imgs)}")
    tv.append("static const uint8_t *const TV_IMG[TV_N] = { "
              + ", ".join(f"IMG{k}" for k in range(len(imgs))) + " };")
    tv.append("static const int32_t TV_D[TV_N] = { "
              + ", ".join(str(im["D"]) for im in imgs) + " };")
    tv.append("static const int TV_PRED[TV_N] = { "
              + ", ".join(str(im["pred"]) for im in imgs) + " };")
    tv.append("")
    with open(os.path.join(out_dir, "test_vectors.h"), "w") as f:
        f.write("\n".join(tv))
    print(f"OK: BUF_A={buf_a} BUF_B={buf_b}, GAP={cout}x{cur}x{cur}, {len(imgs)} test images")


if __name__ == "__main__":
    main()

OK: BUF_A=4096 BUF_B=8192, GAP=29x8x8, 2 test images


In [60]:
#!/usr/bin/env python3
"""Sinh cnn_params.h va test_vectors.h tu thu muc golden_out/ cho cnn.c (Bambu HLS).

Dung:  python3 gen_params.py [golden_out] [out_dir]
"""
import json
import os
import sys


def rd(path):
    with open(path, "r", encoding="utf-8") as f:
        return [int(l.strip(), 16) for l in f if l.strip()]


def sx(v, bits):
    return v - (1 << bits) if v >= (1 << (bits - 1)) else v


def pack(codes):
    """Pack 4-bit codes, 8 per uint32_t (code k at bits [4*(k%8) +: 4] of word k//8)."""
    assert all(0 <= c < 16 for c in codes)
    out = []
    for i in range(0, len(codes), 8):
        w = 0
        for j, c in enumerate(codes[i:i + 8]):
            w |= c << (4 * j)
        out.append(w)
    return out


def emit(lines, ctype, name, vals):
    lines.append(f"static const {ctype} {name}[{len(vals)}] = {{")
    for i in range(0, len(vals), 16):
        lines.append("  " + ", ".join(str(v) for v in vals[i:i + 16]) + ",")
    lines.append("};")
    lines.append("")


def main():
    hex_dir = "golden_out"
    out_dir = "."
    with open(os.path.join(hex_dir, "params.json"), "r", encoding="utf-8") as f:
        p = json.load(f)

    kept_idx = p["kept_idx"]
    n = len(kept_idx)
    assert p["m_bits"] <= 16 and p["bq_bits"] <= 32
    H0 = 64

    head = [
        "/* auto-generated by gen_params.py - do not edit */",
        "#ifndef CNN_PARAMS_H",
        "#define CNN_PARAMS_H",
        "#include <stdint.h>",
        "",
        f"#define IMG_H {H0}",
        f"#define IMG_W {H0}",
        f"#define N_LAYERS {n}",
        f"#define W_FRAC {p['w_frac']}",
        f"#define AMAX {(1 << p['abits']) - 1}",
        f"#define THRESH {p['T']}",
    ]
    body = []
    w_all, m_all, b_all = [], [], []   # one ROM each (fewer separate memories for Bambu)
    buf_a = H0 * H0   # layer-0 input (image) lives in A
    buf_b = 0
    cur = H0
    cout = 1
    for i in range(n):
        cout = len(kept_idx[i])
        cin = 1 if i == 0 else len(kept_idx[i - 1])
        pool = 1 if p["pools"][i] else 0
        nxt = cur // 2 if pool else cur
        in_sz, out_sz = cin * cur * cur, cout * nxt * nxt
        if i % 2 == 0:   # even layer: A -> B
            buf_a, buf_b = max(buf_a, in_sz), max(buf_b, out_sz)
        else:            # odd layer: B -> A
            buf_b, buf_a = max(buf_b, in_sz), max(buf_a, out_sz)

        head += [
            f"#define L{i}_CIN {cin}",
            f"#define L{i}_COUT {cout}",
            f"#define L{i}_H {cur}",
            f"#define L{i}_HW {cur * cur}",
            f"#define L{i}_SH {p['SH'][i]}",
            f"#define L{i}_POOL {pool}",
        ]
        w = rd(os.path.join(hex_dir, f"w_L{i}.hex"))
        assert len(w) == cout * cin * 9, f"w_L{i}: {len(w)} != {cout * cin * 9}"
        m = [sx(v, p["m_bits"]) for v in rd(os.path.join(hex_dir, f"bn_M_L{i}.hex"))]
        b = [sx(v, p["bq_bits"]) for v in rd(os.path.join(hex_dir, f"bn_B_L{i}.hex"))]
        assert len(m) == cout and len(b) == cout
        head += [f"#define L{i}_WOFF {len(w_all)}", f"#define L{i}_MOFF {len(m_all)}"]
        w_all += pack(w)   # packed codes, flat [co][ci][ky][kx], padded to a word per layer
        m_all += m
        b_all += b
        cur = nxt

    head += [f"#define GAP_C {cout}", f"#define GAP_H {cur}", f"#define BUF_A_SIZE {buf_a}", f"#define BUF_B_SIZE {buf_b}", ""]
    fc = rd(os.path.join(hex_dir, "fc_w.hex"))
    assert len(fc) == 2 * cout
    head.append(f"#define FC_WOFF {len(w_all)}")
    w_all += pack(fc)                           # packed codes, flat [class][c]
    emit(body, "uint32_t", "W_ALL", w_all)
    emit(body, "int16_t", "M_ALL", m_all)
    emit(body, "int32_t", "B_ALL", b_all)

    with open(os.path.join(out_dir, "cnn_params.h"), "w") as f:
        f.write("\n".join(head + [""] + body + ["#endif", ""]))

    # test vectors
    tv = ["/* auto-generated by gen_params.py */", "#include <stdint.h>", ""]
    imgs = p["images"]
    for k, im in enumerate(imgs):
        px = rd(os.path.join(hex_dir, f"img{im['idx']}_in.hex"))
        assert len(px) == H0 * H0
        emit(tv, "uint8_t", f"IMG{k}", px)
    tv.append(f"#define TV_N {len(imgs)}")
    tv.append("static const uint8_t *const TV_IMG[TV_N] = { "
              + ", ".join(f"IMG{k}" for k in range(len(imgs))) + " };")
    tv.append("static const int32_t TV_D[TV_N] = { "
              + ", ".join(str(im["D"]) for im in imgs) + " };")
    tv.append("static const int TV_PRED[TV_N] = { "
              + ", ".join(str(im["pred"]) for im in imgs) + " };")
    tv.append("")
    with open(os.path.join(out_dir, "test_vectors.h"), "w") as f:
        f.write("\n".join(tv))
    print(f"OK: BUF_A={buf_a} BUF_B={buf_b}, GAP={cout}x{cur}x{cur}, {len(imgs)} test images")


if __name__ == "__main__":
    main()

OK: BUF_A=4096 BUF_B=8192, GAP=29x8x8, 2 test images
